In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import copy

from sklearn import ensemble
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report

import warnings
warnings.filterwarnings("ignore")

In [25]:
df1 = pd.read_csv('train_data_after.csv', index_col=0)
df2 = pd.read_csv('test_data_after.csv', index_col=0)

In [26]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 157525 entries, 0 to 157524
Data columns (total 10 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CreditScore         157525 non-null  float64
 1   Age                 157525 non-null  float64
 2   Tenure              157525 non-null  float64
 3   Balance             157525 non-null  float64
 4   NumOfProducts       157525 non-null  float64
 5   HasCrCard           157525 non-null  int64  
 6   IsActiveMember      157525 non-null  int64  
 7   Exited              157525 non-null  int64  
 8   Mem__no__Products   157525 non-null  float64
 9   Age_Tenure_product  157525 non-null  float64
dtypes: float64(7), int64(3)
memory usage: 13.2 MB


In [27]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17503 entries, 0 to 17502
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CreditScore         17503 non-null  float64
 1   Age                 17503 non-null  float64
 2   Tenure              17503 non-null  float64
 3   Balance             17503 non-null  float64
 4   NumOfProducts       17503 non-null  float64
 5   HasCrCard           17503 non-null  int64  
 6   IsActiveMember      17503 non-null  int64  
 7   Exited              17503 non-null  int64  
 8   Mem__no__Products   17503 non-null  float64
 9   Age_Tenure_product  17503 non-null  float64
dtypes: float64(7), int64(3)
memory usage: 1.5 MB


In [28]:
train_points = df1.drop(['Exited'], axis=1)
train_values = df1['Exited']
test_points = df2.drop(['Exited'], axis=1)
test_values = df2['Exited']

In [29]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=1500,
    max_depth=16,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight={0: 1.0, 1: 4},   # пробуем сильнее
    max_features='sqrt',
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(train_points, train_values)

# --- Поиск лучшего порога ---
proba = rf_model.predict_proba(test_points)[:, 1]

print("Threshold | Precision | Recall | F1 | Accuracy")
print("-" * 65)

best_f1 = 0
best_threshold = 0.40



for t in np.arange(0.30, 0.55, 0.01):
    pred = (proba >= t).astype(int)
    report = classification_report(test_values, pred, output_dict=True, digits=3)
    p = report['1']['precision']
    r = report['1']['recall']
    f1 = report['1']['f1-score']
    acc = report['accuracy']
    
    print(f"{t:.2f}      | {p:.3f}      | {r:.3f}    | {f1:.3f}   | {acc:.3f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"\nЛучший порог: {best_threshold:.2f} с F1 = {best_f1:.3f}")

Threshold | Precision | Recall | F1 | Accuracy
-----------------------------------------------------------------
0.30      | 0.408      | 0.871    | 0.556   | 0.712
0.31      | 0.414      | 0.864    | 0.559   | 0.718
0.32      | 0.419      | 0.858    | 0.563   | 0.724
0.33      | 0.425      | 0.851    | 0.567   | 0.731
0.34      | 0.431      | 0.842    | 0.570   | 0.737
0.35      | 0.438      | 0.833    | 0.574   | 0.744
0.36      | 0.445      | 0.827    | 0.579   | 0.750
0.37      | 0.450      | 0.818    | 0.581   | 0.755
0.38      | 0.455      | 0.809    | 0.583   | 0.760
0.39      | 0.461      | 0.801    | 0.586   | 0.765
0.40      | 0.467      | 0.795    | 0.588   | 0.770
0.41      | 0.475      | 0.785    | 0.592   | 0.776
0.42      | 0.484      | 0.778    | 0.596   | 0.782
0.43      | 0.492      | 0.769    | 0.600   | 0.788
0.44      | 0.500      | 0.761    | 0.603   | 0.793
0.45      | 0.508      | 0.751    | 0.606   | 0.798
0.46      | 0.517      | 0.743    | 0.610   | 0.803
0.4

In [30]:
importances = rf_model.feature_importances_
feature_names = train_points.columns

# Создаём DataFrame для удобства
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

print("Топ-10 самых важных признаков:")
print(feature_importance_df.head(10))

Топ-10 самых важных признаков:
              Feature  Importance
1                 Age    0.293205
4       NumOfProducts    0.242391
0         CreditScore    0.111191
3             Balance    0.108772
7   Mem__no__Products    0.101053
8  Age_Tenure_product    0.062517
6      IsActiveMember    0.045150
2              Tenure    0.026432
5           HasCrCard    0.009288


Удаляем признками <1% 

In [31]:
del df1['HasCrCard']
del df2['HasCrCard']


In [32]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 157525 entries, 0 to 157524
Data columns (total 9 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   CreditScore         157525 non-null  float64
 1   Age                 157525 non-null  float64
 2   Tenure              157525 non-null  float64
 3   Balance             157525 non-null  float64
 4   NumOfProducts       157525 non-null  float64
 5   IsActiveMember      157525 non-null  int64  
 6   Exited              157525 non-null  int64  
 7   Mem__no__Products   157525 non-null  float64
 8   Age_Tenure_product  157525 non-null  float64
dtypes: float64(7), int64(2)
memory usage: 12.0 MB


In [33]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 17503 entries, 0 to 17502
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   CreditScore         17503 non-null  float64
 1   Age                 17503 non-null  float64
 2   Tenure              17503 non-null  float64
 3   Balance             17503 non-null  float64
 4   NumOfProducts       17503 non-null  float64
 5   IsActiveMember      17503 non-null  int64  
 6   Exited              17503 non-null  int64  
 7   Mem__no__Products   17503 non-null  float64
 8   Age_Tenure_product  17503 non-null  float64
dtypes: float64(7), int64(2)
memory usage: 1.3 MB


In [34]:
rf_model = RandomForestClassifier(
    n_estimators=1200,
    max_depth=18,
    min_samples_split=5,
    min_samples_leaf=2,
    class_weight={0: 1.0, 1: 3.8},   # подобранный вес
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

print("Обучение Random Forest...")
rf_model.fit(train_points, train_values)

# ==================== 2. Предсказание с порогом 0.55 ====================
# Получаем вероятности класса 1
proba = rf_model.predict_proba(test_points)[:, 1]

# Применяем выбранный порог 0.55
THRESHOLD = 0.55
y_pred = (proba >= THRESHOLD).astype(int)

# ==================== 3. Вывод результатов ====================
print(f"\nClassification Report при пороге {THRESHOLD}:")
print(classification_report(test_values, y_pred, digits=3))

Обучение Random Forest...

Classification Report при пороге 0.55:
              precision    recall  f1-score   support

           0      0.900     0.895     0.898     13878
           1      0.608     0.620     0.614      3625

    accuracy                          0.838     17503
   macro avg      0.754     0.758     0.756     17503
weighted avg      0.840     0.838     0.839     17503

